In [ ]:
%load_ext autoreload
%autoreload 2

from tabulate import tabulate

from fart.constants import MAGNITUDE
from fart.features.align_predicted_signal import align_predicted_signal
from fart.features.calculate_trade_returns import calculate_trade_returns
from fart.model.cnn_builder import CNNBuilder
from fart.model.cnn_config import CNNConfig
from fart.model.evaluate_model import evaluate_model
from fart.model.prepare_datasets import prepare_datasets
from fart.model.train_model import train_model
from fart.utils import get_data_filepath, get_project_root
from fart.visualization.evaluation_line_chart import evaluation_line_chart
from fart.visualization.learning_curve_chart import learning_curve_chart
from fart.visualization.plot_styles import apply_plot_styles
from fart.visualization.predicted_vs_actual_scatter import predicted_vs_actual_scatter
from fart.visualization.trade_returns import plot_trade_returns

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
market = "BTC-EUR"
interval = "1d"
data_filepath = get_data_filepath(assets_dir, market, interval)

cnn_config = CNNConfig(
    num_lags=100,
    num_blocks=3,
    num_channels=32,
    kernel_size=5,
)
batch_size = 16
learning_rate = 0.001
num_epochs = 100
train_size = 0.6
val_size = 0.2

In [ ]:
x_train, y_train, x_val, y_val, x_test, y_test = prepare_datasets(
    data_filepath=data_filepath,
    target=MAGNITUDE,
    num_lags=cnn_config.num_lags,
    train_size=train_size,
    val_size=val_size,
)

In [ ]:
model, history = train_model(
    model=CNNBuilder(cnn_config).build(),
    x_train=x_train,
    y_train=y_train,
    x_val=x_val,
    y_val=y_val,
    batch_size=batch_size,
    learning_rate=learning_rate,
    num_epochs=num_epochs,
)

In [ ]:
learning_curve_chart(history)

In [ ]:
(
    y_train_pred,
    y_test_pred,
    accuracy_train,
    accuracy_test,
    rmse_train,
    rmse_test,
    mae_train,
    mae_test,
) = evaluate_model(
    model=model,
    x_train=x_train,
    y_train=y_train,
    x_test=x_test,
    y_test=y_test,
)

print(
    tabulate(
        [
            [
                "Train",
                round(accuracy_train, 2),
                rmse_train,
                mae_train,
            ],
            [
                "Test",
                round(accuracy_test, 2),
                rmse_test,
                mae_test,
            ],
        ],
        headers=["Dataset", "Accuracy", "RMSE", "MAE"],
    )
)

In [ ]:
evaluation_line_chart(
    y_train=y_train,
    y_test=y_test,
    y_pred=y_test_pred,
)

In [ ]:
evaluation_line_chart(
    y_test=y_test,
    y_pred=y_test_pred,
)

In [ ]:
predicted_vs_actual_scatter(
    y_test=y_test,
    y_pred=y_test_pred,
)

In [ ]:
returns, profits = calculate_trade_returns(
    magnitudes=y_test,
    predicted_magnitudes=align_predicted_signal(y_test_pred),
)

plot_trade_returns(returns, profits)